## 3-hop architecture: Bronze is source, Silver is fact/dims, & Gold is summary tables

* we had seen in previous sections the importance of dimensional modeling and why we need to store a copy of the raw data for replayability (in bronze schema)
* The 3-hop architecture is the idea of transforming data in a series of layers to make the data easy to use by stakeholders
* The key idea where you store the raw data -> Model it for analytics -> join and group by and get it ready for BI/Analysts is a few decades old
* Before databricks popularized medallion, every company had its own naming conventions. raw/base -> modelled, etc
* Most companies recommend following a pattern, such as Databrick's medallion and dbt's staging -> Intermediate -> Marts pattern
* While Databricks and dbt recommend their way, here is what I have seen work really well in most companies
  - `bronze`: raw data as is from external system with data type and naming changes (bronze.customer, bronze.customer_address, etc)
  - `silver`: data modelled as facts and dimensions based on Kimball data modeling (e.g. fact_order, dim_user, etc)
  - `gold`: uses only silver layer tables to create tables for use by BI/Analysts, this is meant to reduce the cognitive load for end users. They do not have to worry about the join criteria and metric computations.

![Medallion](images/bronze_silver_gold.png)

#### Exercise [30 min]

Based on the image above and how multi hop architecture are supposed to work, Create folders in this directory called `bronze`, `silver`, & `gold`. 

In the `bronze` folder create the following Python scripts:

1. product.py
2. customer.py
3. customer_address.py
4. order.py
5. order_lines.py
6. payment.py

These should follow the format of 

```python
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}
TABLE_NAME = "local.bronze.product"


def run(spark: SparkSession) -> None:
    spark.read.jdbc(
        url=JDBC_URL,
        table=f"""(
            SELECT *
            FROM public.product
        ) product""",
        properties=JDBC_PROPERTIES,
    ).writeTo(TABLE_NAME).createOrReplace()

if __name__ == '__init__':
    spark = SparkSession.builder.appName(TABLE_NAME).master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    run(spark)
```

In the `silver` folder create the following Python scripts (which we had created in the previous sections):

1. dim_customer.py (use snapshot)
2. dim_product.py (use SCD2)
3. fct_orders.py (incremental)
4. fct_order_lines.py (incremental)

These should follow the format of 

```python
import argparse
from pyspark.sql import DataFrame, SparkSession

TABLE_NAME = "local.silver.dim_customer"

def extract(spark: SparkSession) -> dict[str, DataFrame]:
    customer_df = spark.table("local.bronze.customer")
    customer_address_df = spark.table("local.bronze.customer_address")
    return {"customer_df": customer_df, "customer_address_df": customer_address_df}


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    input_dfs["customer_df"].createOrReplaceTempView("customer")
    input_dfs["customer_address_df"].createOrReplaceTempView("customer_address")

    return spark.sql("""
        SELECT
            c.customer_id,
            c.email,
            c.full_name,
            c.phone,
            c.status,
            c.created_at,
            c.updated_at,
            COLLECT_LIST(
                STRUCT(
                    ca.is_default,
                    CONCAT(ca.line1, ', ', ca.city, ', ', ca.state, ', ', ca.country) AS address
                )
            ) AS addresses
        FROM customer c
        LEFT JOIN customer_address ca USING (customer_id)
        GROUP BY 1, 2, 3, 4, 5, 6, 7
    """)


def load(output_df: DataFrame) -> None:
    output_df.writeTo(TABLE_NAME).createOrReplace()


def run(spark: SparkSession) -> None:
    load(transform(extract(spark)))

if __name__ == '__init__':
    parser = argparse.ArgumentParser(description=f"{TABLE_NAME} ETL")
    parser.add_argument(
        "--start-time",
        required=True,
        help="Start time (inclusive), format: YYYY-MM-DD HH:MM:SS",
    )
    parser.add_argument(
        "--end-time",
        required=True,
        help="End time (exclusive), format: YYYY-MM-DD HH:MM:SS",
    )
    args = parser.parse_args()

    spark = SparkSession.builder.appName(TABLE_NAME).master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    run(spark) # add start and end time for incremental pipelines
```



In [ ]:
# Start a SparkSession
from pyspark.sql import Row, SparkSession

spark = (
    SparkSession.builder.appName("03_medallion_arch").master("local[*]").getOrCreate()
)

In [ ]:
# Delete bronze tables if it exists
spark.sql("DROP TABLE IF EXISTS local.bronze.product")
spark.sql("DROP TABLE IF EXISTS local.bronze.customer")
spark.sql("DROP TABLE IF EXISTS local.bronze.customer_address")
spark.sql("DROP TABLE IF EXISTS local.bronze.orders")
spark.sql("DROP TABLE IF EXISTS local.bronze.order_lines")
spark.sql("DROP TABLE IF EXISTS local.bronze.payment")

# Delete silver tables if it exists
spark.sql("DROP TABLE IF EXISTS local.bronze.dim_product")
spark.sql("DROP TABLE IF EXISTS local.bronze.dim_customer")
spark.sql("DROP TABLE IF EXISTS local.bronze.fct_orders")
spark.sql("DROP TABLE IF EXISTS local.silver.fct_order_lines")

In [ ]:
%%bash
%%capture
echo 'RUN Bronze Tables'
echo '================='

echo 'RUN Bronze local.bronze.product'
echo '================='
uv run python ./bronze/product.py

echo 'RUN Bronze local.bronze.customer'
echo '================='
uv run python ./bronze/customer.py

echo 'RUN Bronze local.bronze.customer_address'
echo '================='
uv run python ./bronze/customer_address.py

echo 'RUN Bronze local.bronze.orders.py'
echo '================='
uv run python ./bronze/orders.py

echo 'RUN Bronze local.bronze.order_lines'
echo '================='
uv run python ./bronze/order_lines.py

echo 'RUN Bronze local.bronze.payment.py'
echo '================='
uv run python ./bronze/payment.py


In [ ]:
spark.table("local.bronze.order_lines").limit(2).toPandas()

In [ ]:
%%bash
%%capture
echo 'RUN Silver Tables'
echo '================='

echo 'RUN Silver local.silver.dim_customer'
echo '================='
uv run python ./silver/dim_customer.py

echo 'RUN Silver local.silver.dim_product'
echo '================='
uv run python ./silver/dim_product.py

echo 'RUN Silver local.silver.fct_orders'
echo '================='
uv run python ./silver/fct_orders.py --start-time '2025-01-01 00:00:00' --end-time '2025-02-01 00:00:00'

echo 'RUN Silver local.silver.fct_order_lines'
echo '================='
uv run python ./silver/fct_order_lines.py --start-time '2025-01-01 00:00:00' --end-time '2025-02-01 00:00:00'


In [ ]:
spark.table("local.silver.fct_order_lines").limit(2).toPandas()

## Gold tables are for select * from gold_tbl by end users

* Gold layer tables are for querying by BI/ Analysts

* The idea is to reduce the barrier to data usage by end users

* Gold layer aims to reduce the number of joins, group bys, and metric creation

* End users may make subtle changes (with joins/group by/metrics) that significantly alter their data

* An option most companies use is a tool like lookml or DAX to define the dimensions, facts, and metrics and enable end-users to visually compose their query 

* Visual tools are hard to maintain and extremely expensive (as they often involve recomputing the same query for multiple end users)

* Visual tools are also harder to debug issues with metrics.

#### Example

![OBT -> Summary Tables](images/gold_obt_summary.png)

* As we see from the above there are 2 main types of gold tables: 

  - One big table (OBT) formed by taking a fact table and left-joining all of the available dimensions to it

  - Summary (or pre_agg) table, which is an aggregate of different grains created from the OBT table.

* One Big Table (OBT) eliminates joins and makes creating summary tables just a group by.

* OBT comes at the expense of storage efficiency and increased maintenance complexity. 

* Example: Sales OBT with transaction details plus customer name, product description, and store location all in one table.

* Summary/Aggregate tables are pre-calculated tables storing rolled-up metrics at higher levels of granularity than base fact tables. 

* Improves query performance for common reporting patterns by avoiding expensive aggregation calculations at runtime. Monthly sales summary by region instead of querying daily transaction details each time.

#### Exercise [10 min]

* Consider a `fct_order_lines` item table, and here are the queries written by 2 stakeholders to calculate average order value (AOV). 
* Is this correct? If not, What is wrong with it? 

In [ ]:
spark.sql("""
    SELECT
        DATE_TRUNC('week', created_at)  AS week,
        ROUND(AVG(order_total), 2)      AS avg_order_value
    FROM (
        SELECT
            order_id,
            DATE_TRUNC('week', created_at) AS created_at,
            SUM(line_total)                AS order_total
        FROM local.silver.fct_order_lines
        GROUP BY order_id, DATE_TRUNC('week', created_at)
    )
    GROUP BY week
    ORDER BY week
""").toPandas()

In [ ]:
spark.sql("""
    SELECT
        DATE_TRUNC('week', created_at)              AS week,
        ROUND(AVG(unit_price * quantity - discount_amt), 2) AS avg_order_value
    FROM local.silver.fct_order_lines
    GROUP BY week
    ORDER BY week
""").toPandas()

* Solution:
  
* query 2 computes the average but groups by week, which means it computes the average at a line level, which is incorrect

| order_id | order_line_id | amount |
|----------|---------|--------|
| A        | 1    | 10     |
| A        | 2    | 90     |
| B        | 1    | 50     |

```text
-- Query1: AVG(SUM(amount)) grouped by order + week and they group by week → true AOV
order A = 10 + 90 = 100
order B = 50      = 50
AVG(100, 50)      = 75

-- Query2: AVG(amount) grouped by week → wrong AOV
AVG(10, 90, 50)   = 50
```

* Note: the gold layer may not always be necessary, especially if your stakeholders are few; get the silver layer right first
* We need to be aware of additive and non-additive numerical columns.
* An additive column is one that can be aggregated and still retain its meaning, such as the sum of quantity, price, etc.
* A non-additive column is one that cannot be aggregated: IDs, numerical codes, ratios, distinct counts, etc.

## Use nested data structures to create wide OBTs

* One big table needs as much information as it can store. This will ensure that the summary tables we create are grouped by the right grain and defined with the right metric.
* Left-joining all dimensions to the OBT fact table is straightforward, but across multiple tables, identifying the appropriate columns/naming collisions can be challenging.

#### Exercise [5 min]

* Assume you are left-joining customer, seller, warehouse, etc., dimensions to a `fct_order_lines` table, and that your OBT table ends up with 100s of columns. 
* How will you ensure that the table is easy to use and intuitive? How will a customer’s address be easy to distinguish from a seller’s address in this table?


**Solution**

* One simple solution is to prepend column names, such as customer_address or seller_address.
* We can also use nested data structures—for example, arrays and structs—to represent dimensional attributes within an OBT.
* But we also need to ensure schema evolution so that the nested data structure can evolve without too much manual changes.

#### Example

Let’s consider the tables.

1. `order_line`
2. `product_variant`

and create an OBT from it

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}
TABLE_NAME = "local.bronze.product_variant"


def run(spark: SparkSession) -> None:
    spark.read.jdbc(
        url=JDBC_URL,
        table=f"""(
            SELECT *
            FROM public.product_variant
        ) product_variant""",
        properties=JDBC_PROPERTIES,
    ).writeTo(TABLE_NAME).createOrReplace()

In [ ]:
# Create bronze.product_variant
spark.sparkContext.setLogLevel("ERROR")
run(spark)

In [ ]:
spark.sql("""
    WITH order_line_obt AS (
        SELECT
            ol.order_line_id,
            ol.order_id,
            ol.variant_id,
            ol.quantity,
            ol.unit_price,
            ol.discount_amt,
            ol.line_total,
            ol.created_at,
            ol.updated_at,
            STRUCT(
                pv.variant_id,
                pv.product_id,
                pv.sku,
                pv.size,
                pv.color,
                pv.price_override,
                pv.weight_kg,
                pv.is_active,
                pv.created_at,
                pv.updated_at
            ) AS product_variant
        FROM local.bronze.order_lines ol
        LEFT JOIN local.bronze.product_variant pv USING (variant_id)
    )
    SELECT
        product_variant.is_active,
        *
    FROM order_line_obt
    LIMIT 2
""").toPandas()

* Notice how we put all our product_variant information in a STRUCT
* This enables use to select their attributes more ergonomically `product_variant.attribute_name`

#### Exercise [10 min]

Assume you want to enrich the `order_line_obt` with `product_attributes`. How would you model the `order_line_obt` table?

*Note* The product_variant -> product_attribute is 1:Many

**Hint**: Look at [COLLECT_LIST](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.collect_list.html) and [STRUCT](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.struct.html) function docs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}
TABLE_NAME = "local.bronze.product_attribute"


def run(spark: SparkSession) -> None:
    spark.read.jdbc(
        url=JDBC_URL,
        table=f"""(
            SELECT *
            FROM public.product_attribute
        ) product_attribute""",
        properties=JDBC_PROPERTIES,
    ).writeTo(TABLE_NAME).createOrReplace()

In [ ]:
run(spark)

In [ ]:
spark.sql("""
    WITH pa_agg AS (
        SELECT
            variant_id,
            COLLECT_LIST(
                STRUCT(
                    attribute_id,
                    attr_key,
                    attr_value,
                    display_order
                )
            ) AS attributes
        FROM local.bronze.product_attribute
        GROUP BY variant_id
    ),
    order_line_obt AS (
        SELECT
            ol.order_line_id,
            ol.order_id,
            ol.variant_id,
            ol.quantity,
            ol.unit_price,
            ol.discount_amt,
            ol.line_total,
            ol.created_at,
            ol.updated_at,
            STRUCT(
                pv.variant_id,
                pv.product_id,
                pv.sku,
                pv.size,
                pv.color,
                pv.price_override,
                pv.weight_kg,
                pv.is_active,
                pv.created_at,
                pv.updated_at,
                COALESCE(pa.attributes, ARRAY()) AS attributes
            ) AS product_variant
        FROM local.bronze.order_lines ol
        LEFT JOIN local.bronze.product_variant pv USING (variant_id)
        LEFT JOIN pa_agg pa USING (variant_id)
    )
    SELECT
        order_line_id,
        product_variant.is_active  AS is_active,
        product_variant.attributes AS attributes,
        product_variant.attributes[0] AS attributes_first
    FROM order_line_obt
    LIMIT 3
""").toPandas()

* In this case we can use an `Struct[Array[Struct]]` to represent attributes within a variant. We are trading off convenience of storage for complexity of querying.

* When creating pre_aggregates tables you'd often need to compute distinct counts, this is easy to do with query, but hard to model
* A pattern to use here is called `Lness:  recency-style window metrics`  computing distinct counts over a trailing window (last L days/weeks).
* Let's look at an example

In [ ]:
spark.sql("""
    SELECT DISTINCT
        DATE_TRUNC('day', created_at)                           AS day,
        approx_count_distinct(order_id)
            OVER (
                ORDER BY DATE_TRUNC('day', created_at)
                RANGE BETWEEN INTERVAL '7' DAYS PRECEDING
                          AND CURRENT ROW
            )                                                   AS orders_last_7d,
        approx_count_distinct(order_id)
            OVER (
                ORDER BY DATE_TRUNC('day', created_at)
                RANGE BETWEEN INTERVAL '30' DAYS PRECEDING
                          AND CURRENT ROW
            )                                                   AS orders_last_30d,
        approx_count_distinct(order_id)
            OVER (
                ORDER BY DATE_TRUNC('day', created_at)
                RANGE BETWEEN INTERVAL '90' DAYS PRECEDING
                          AND CURRENT ROW
            )                                                   AS orders_last_90d 
    FROM local.bronze.order_lines
    ORDER BY 1 DESC
""").limit(10).toPandas()

* **Note** we use approx distinct count since count(distinct) is not yet supported https://issues.apache.org/jira/browse/SPARK-30212


## Run fact pipelines hourly for data availability and daily to catch late events (aka Lambda Architecture)

* Most fact pipelines use a lambda architecture, where the pipeline will run with a smaller time range frequently and longer timerange at a lower frequency.

#### Example

* We see that `fct_order_lines` have late arrving data, where data can arrive upto 20h late.
* Our stakeholders want `fct_order_lines` data in under 4 hours for analytics.
* We can run `fct_order_lines` every 1h with 1h window and every day with 24h window.
* These are called speed/batch runs. A batch can also be called catch-up/look-back, etc.
* This is called a lambda architecture.

![Lambda Architecture](images/lambda_architecture.png)

#### Exercise [5 min]

* If the pipeline script `./silver/fct_orders.py` runs at
  1. Day 1 10 PM
  2. Day 1 11 PM
  3. Day 2 12 AM (next day)
  4. Day 2 1 AM (next day)
* Assume 12AM is the batch run, what would its start and end time be?


**solution**

created_at >= Day 1 12:00:00 and created_at < Day 2 12:00:00

## Recap

In this section, we covered data flow patterns. We saw 

1. What medallion architecture is and how it helps companies standardize on a pattern
2. How Gold tables make it easy for end-users to use the data that you create
3. How nested data, lness techniques, & Lambda architecture help team keeps data clean and *up-to-date*